# Amazon rivers-on freshwater–dye coupling (80 days)

Rivers-on only. Quantifies how strongly the passive OAE proxy remains associated with the Amazon freshwater plume. Results after day 30 are exploratory because the WENO salinity run develops a growing high-salinity overshoot near the river mouth.

In [ ]:
using Oceananigans
using CairoMakie
using Printf

data_directory = raw"C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\new jld2 files\no spinups\80 day salinity dye"
run_prefix = "amazon_rivers_on_validation_FINAL_nz20_gm_off_redi_off_salinity_80day_v2"
dye_file = joinpath(data_directory, "amazon_rivers_on_80d_dye.jld2")
salinity_file = joinpath(data_directory, "amazon_rivers_on_80d_salinity_3d.jld2")

for file in (dye_file, salinity_file)
    @assert isfile(file) "Missing input file: $file. Update data_directory if the 80-day files were moved."
end

dye = FieldTimeSeries(dye_file, "dye"; backend=OnDisk())
salinity = FieldTimeSeries(salinity_file, "S"; backend=OnDisk())
@assert length(dye.times) == length(salinity.times)
days_saved = Float64.(dye.times) ./ 86400
println("Loaded $(length(days_saved)) paired dye/salinity outputs through day $(last(days_saved)).")

In [ ]:
longitude, latitude, depth = nodes(dye.grid, Center(), Center(), Center())
longitude_faces, latitude_faces, depth_faces = nodes(dye.grid, Face(), Face(), Face())
earth_radius = 6.371e6

delta_longitude = diff(deg2rad.(longitude_faces))
delta_sin_latitude = diff(sin.(deg2rad.(latitude_faces)))
delta_depth = diff(depth_faces)
cell_volume = earth_radius^2 .*
              reshape(delta_longitude, length(delta_longitude), 1, 1) .*
              reshape(delta_sin_latitude, 1, length(delta_sin_latitude), 1) .*
              reshape(delta_depth, 1, 1, length(delta_depth))

initial_salinity = Array(interior(salinity[1]))
wet_cell = isfinite.(initial_salinity) .& (initial_salinity .> 0)
upper_100m = wet_cell .& reshape(depth .>= -100, 1, 1, length(depth))

ocean_salinity_reference = 35.0
river_salinity_reference = 0.0
plume_thresholds = (30.0, 32.0, 34.0)

In [ ]:
number_of_times = length(days_saved)
fraction_inside_plume = zeros(number_of_times, length(plume_thresholds))
upper_100m_fraction_inside_plume = zeros(number_of_times, length(plume_thresholds))
dye_weighted_freshwater_fraction = zeros(number_of_times)
upper_100m_dye_weighted_freshwater_fraction = zeros(number_of_times)
salinity_minimum = zeros(number_of_times)
salinity_maximum = zeros(number_of_times)

for n in eachindex(days_saved)
    concentration = Float64.(Array(interior(dye[n])))
    salinity_field = Float64.(Array(interior(salinity[n])))

    concentration[.!isfinite.(concentration)] .= 0
    concentration[.!wet_cell] .= 0

    dye_mass = concentration .* cell_volume
    total_dye_mass = sum(dye_mass)
    upper_dye_mass = sum(dye_mass[upper_100m])

    freshwater_fraction = clamp.(
        (ocean_salinity_reference .- salinity_field) ./
        (ocean_salinity_reference - river_salinity_reference),
        0,
        1
    )

    dye_weighted_freshwater_fraction[n] = sum(dye_mass .* freshwater_fraction) / total_dye_mass
    upper_100m_dye_weighted_freshwater_fraction[n] = sum((dye_mass .* freshwater_fraction)[upper_100m]) / upper_dye_mass

    for (m, threshold) in enumerate(plume_thresholds)
        plume_mask = wet_cell .& (salinity_field .< threshold)
        fraction_inside_plume[n, m] = sum(dye_mass[plume_mask]) / total_dye_mass
        upper_plume_mask = upper_100m .& (salinity_field .< threshold)
        upper_100m_fraction_inside_plume[n, m] = sum(dye_mass[upper_plume_mask]) / upper_dye_mass
    end

    salinity_minimum[n] = minimum(salinity_field[wet_cell])
    salinity_maximum[n] = maximum(salinity_field[wet_cell])
end

In [ ]:
figure = Figure(size=(1200, 950))

plume_axis = Axis(figure[1, 1], xlabel="Simulation day", ylabel="Fraction of total dye", title="Dye remaining inside salinity-defined plume")
for (m, threshold) in enumerate(plume_thresholds)
    lines!(plume_axis, days_saved, fraction_inside_plume[:, m], linewidth=3, label="S < $(threshold) PSU")
end
vspan!(plume_axis, 30, maximum(days_saved); color=(:orange, 0.10))
axislegend(plume_axis, position=:rt)

upper_axis = Axis(figure[1, 2], xlabel="Simulation day", ylabel="Fraction of upper-100-m dye", title="Upper-100-m plume coupling")
for (m, threshold) in enumerate(plume_thresholds)
    lines!(upper_axis, days_saved, upper_100m_fraction_inside_plume[:, m], linewidth=3, label="S < $(threshold) PSU")
end
vspan!(upper_axis, 30, maximum(days_saved); color=(:orange, 0.10))
axislegend(upper_axis, position=:rt)

freshwater_axis = Axis(figure[2, 1], xlabel="Simulation day", ylabel="Dye-weighted freshwater fraction", title="Freshwater association of the dye")
lines!(freshwater_axis, days_saved, dye_weighted_freshwater_fraction, linewidth=3, label="All depths")
lines!(freshwater_axis, days_saved, upper_100m_dye_weighted_freshwater_fraction, linewidth=3, label="Upper 100 m")
vspan!(freshwater_axis, 30, maximum(days_saved); color=(:orange, 0.10))
axislegend(freshwater_axis, position=:rt)

quality_axis = Axis(figure[2, 2], xlabel="Simulation day", ylabel="Salinity (PSU)", title="Salinity quality control")
lines!(quality_axis, days_saved, salinity_minimum, linewidth=3, label="Minimum")
lines!(quality_axis, days_saved, salinity_maximum, linewidth=3, label="Maximum")
hlines!(quality_axis, [40], color=:red, linestyle=:dash, label="40 PSU warning")
vspan!(quality_axis, 30, maximum(days_saved); color=(:orange, 0.10))
axislegend(quality_axis, position=:lt)

Label(figure[3, 1:2], "Orange shading marks the lower-confidence period after day 30 in this WENO-salinity run.", color=:darkorange)
figure

In [ ]:
requested_days = [0, 10, 30, 60, 80]
selected_indices = [argmin(abs.(days_saved .- day)) for day in requested_days]
map_figure = Figure(size=(1500, 350))

for (panel, index) in enumerate(selected_indices)
    surface_dye = Float64.(Array(interior(dye[index])))[:, :, end]
    surface_salinity = Float64.(Array(interior(salinity[index])))[:, :, end]
    surface_wet = wet_cell[:, :, end]
    surface_dye[.!surface_wet] .= NaN
    surface_salinity[.!surface_wet] .= NaN

    axis = Axis(map_figure[1, panel], xlabel="Longitude", ylabel=panel == 1 ? "Latitude" : "", title="Day $(round(days_saved[index], digits=1))")
    heatmap!(axis, longitude, latitude, log10.(max.(surface_dye, 1e-12)); colorrange=(-6, 0), colormap=:viridis)
    contour!(axis, longitude, latitude, surface_salinity; levels=collect(plume_thresholds), color=[:white, :cyan, :orange], linewidth=2)
end
Colorbar(map_figure[1, 6], limits=(-6, 0), colormap=:viridis, label="log₁₀ surface dye")
map_figure